In [1]:
"""
《常暗之厢》模组解析工作流。
核心逻辑：加载文档 → 解析场景/事件 → 需求匹配 → 交叉验证 → 文学性扩充。
"""
import sys
import json

# 将 src/ 加入路径以导入依赖模块
sys.path.insert(0, "../src")

from utils import parser, estimate_and_truncate_context
from parsers import parse_scenes_from_document, parse_events_from_document
from pipeline import resolve_requirements, cross_validate_and_revise, expand_scene_descriptions
from llm import call_deepseek_summarize

In [2]:
content = parser("../常暗之厢（7版规则，简体修正版）.docx")
content = estimate_and_truncate_context(content)

[Token 预估] content: 12,458 tokens
[Token 预估] 合计: 12,458 tokens (上限: 300,000)
[Token 预估] 无需截断，直接使用原文


In [4]:
res_scenes = parse_scenes_from_document(content=content)

In [5]:
res_event = parse_events_from_document(content=content)

In [ ]:
# 保存结果到 JSON 文件
with open("../data/output/scene_output.json", "w", encoding="utf-8") as f:
    json.dump(res_scenes, f, ensure_ascii=False, indent=2)

print(f"已保存至 data/output/scene_output.json，共 {len(res_scenes)} 个场景")

In [ ]:
# 保存结果到 JSON 文件
with open("../data/output/res_event.json", "w", encoding="utf-8") as f:
    json.dump(res_event, f, ensure_ascii=False, indent=2)

print(f"已保存至 data/output/res_event.json，共 {len(res_event)} 个场景")

In [ ]:
summary = call_deepseek_summarize(
    content,
    max_chars=1000,
    focus="故事整体的背景和氛围",
    output_path="../data/summary.txt"
)

In [ ]:
result = resolve_requirements(
    events_path="../data/output/res_event.json",
    scenes_path="../data/output/scene_output.json",
    content=content
    )

In [ ]:
result = cross_validate_and_revise(
    events_path="../data/output/res_event_resolved.json",
    scenes_path="../data/output/scene_output_resolved.json",
    content=content,
    auto_revise=True,
)

In [ ]:
expanded_scenes = expand_scene_descriptions(
    scenes_path="../data/output/scene_output_revised.json",
    events_path="../data/output/res_event_revised.json",
    content=content,
    output_path="../data/output/scene_output_expanded.json",
)